
#### 철새 데이터
##### Silver: raw.bird → silver.bird_observation
##### coords[i] + features[i]를 row로 펼치고 파생 컬럼 전부 계산



In [ ]:
import sys
sys.path.append("/Workspace/방역로/00_Shared_Utils")
from utils_config import CATALOG

from pyspark.sql import functions as F, Window
from pyspark.sql.types import *

In [ ]:
# raw 테이블
RAW_BIRD = spark.table("{CATALOG}.raw.bird")

# silver 테이블
SILVER_BIRD = f"{CATALOG}.silver.bird_observation"

SILVER_BIRD_TEST = f"{CATALOG}.silver.bird_test"
  # 검증용, 완료 후 제거

In [ ]:
bird_raw = spark.table(f"{CATALOG}.raw.bird") \
    .filter(F.col("meta.err") == False) \
    .withColumn("recv_id",      F.col("data.info.recv_id")) \
    .withColumn("taxon_nm",     F.col("data.info.taxon_nm")) \
    .withColumn("visitant_sno", F.col("data.info.visitant_sno")) \
    .withColumn("coords",       F.col("data.coords")) \
    .withColumn("features",     F.col("data.features"))

In [ ]:
bird_flat = bird_raw \
    .withColumn("zipped", F.arrays_zip("coords", "features")) \
    .select("recv_id", "taxon_nm", "visitant_sno",
            F.posexplode("zipped").alias("idx", "z")) \
    .select(
        "recv_id", "taxon_nm", "visitant_sno",
        (F.col("idx") + 1).alias("point_order"),
        F.col("z.coords")[0].alias("longitude"),
        F.col("z.coords")[1].alias("latitude"),
        F.col("z.features")[0].alias("observed_timestamp_ms"),
        F.col("z.features")[1].alias("feature_1"),
        F.col("z.features")[2].alias("feature_2"),
    )

bird_flat.limit(3).display()

In [ ]:
# 셀 4 - 타임스탬프
bird_ts = bird_flat \
    .withColumn("observed_at_utc",
        F.to_timestamp(F.col("observed_timestamp_ms") / 1000)) \
    .withColumn("observed_at_kst",
        F.from_utc_timestamp("observed_at_utc", "Asia/Seoul")) \
    .withColumn("observed_date_kst",      F.to_date("observed_at_kst")) \
    .withColumn("observed_year_kst",      F.year("observed_at_kst")) \
    .withColumn("observed_month_kst",     F.month("observed_at_kst")) \
    .withColumn("observed_year_month_kst",
        F.date_trunc("month", "observed_at_kst").cast(DateType())) \
    .withColumn("observed_weekday_kst",   F.date_format("observed_at_kst", "EEEE")) \
    .withColumn("observed_hour_kst",      F.hour("observed_at_kst"))

bird_ts.limit(1).display()

In [ ]:
# 셀 5 - 윈도우
w_track = Window.partitionBy("recv_id", "visitant_sno").orderBy("point_order")
w_full  = Window.partitionBy("recv_id", "visitant_sno") \
                .rowsBetween(Window.unboundedPreceding, Window.unboundedFollowing)
w_day   = Window.partitionBy("recv_id", "visitant_sno", "observed_date_kst")
w_month = Window.partitionBy("recv_id", "visitant_sno", "observed_year_month_kst")

In [ ]:
# 셀 6 - 이전 포인트
bird_seg = bird_ts \
    .withColumn("prev_point_order", F.lag("point_order", 1, 0).over(w_track)) \
    .withColumn("from_longitude",
        F.coalesce(F.lag("longitude", 1).over(w_track), F.col("longitude"))) \
    .withColumn("from_latitude",
        F.coalesce(F.lag("latitude",  1).over(w_track), F.col("latitude"))) \
    .withColumn("_from_ts_ms", F.lag("observed_timestamp_ms", 1).over(w_track)) \
    .withColumn("to_longitude", F.col("longitude")) \
    .withColumn("to_latitude",  F.col("latitude"))

bird_seg.limit(1).display()

In [ ]:
# 셀 7 - Haversine + 속도
def haversine(lat1, lon1, lat2, lon2):
    dlat = F.radians(F.col(lat2) - F.col(lat1))
    dlon = F.radians(F.col(lon2) - F.col(lon1))
    a = F.sin(dlat/2)**2 + F.cos(F.radians(F.col(lat1))) * F.cos(F.radians(F.col(lat2))) * F.sin(dlon/2)**2
    return F.lit(2 * 6371.0) * F.asin(F.sqrt(a))

bird_dist = bird_seg \
    .withColumn("distance_from_prev_km",
        F.when(F.col("prev_point_order") == 0, F.lit(0.0))
         .otherwise(haversine("from_latitude", "from_longitude", "to_latitude", "to_longitude"))) \
    .withColumn("time_diff_from_prev_hours",
        F.when(F.col("_from_ts_ms").isNull(), F.lit(0.0))
         .otherwise((F.col("observed_timestamp_ms") - F.col("_from_ts_ms")) / 3_600_000.0)) \
    .withColumn("speed_from_prev_kmh",
        F.when(F.col("time_diff_from_prev_hours") == 0, F.lit(0.0))
         .otherwise(F.col("distance_from_prev_km") / F.col("time_diff_from_prev_hours"))) \
    .drop("_from_ts_ms")

bird_dist.limit(3).display()

In [ ]:
# 셀 8 - 트랙 전체 집계
bird_agg = bird_dist \
    .withColumn("point_count",             F.count("*").over(w_full)) \
    .withColumn("segment_count",           F.count("*").over(w_full) - 1) \
    .withColumn("total_distance_km",       F.sum("distance_from_prev_km").over(w_full)) \
    .withColumn("max_segment_distance_km", F.max("distance_from_prev_km").over(w_full)) \
    .withColumn("avg_segment_distance_km",
        F.when(F.col("segment_count") == 0, F.lit(0.0))
         .otherwise(F.col("total_distance_km") / F.col("segment_count"))) \
    .withColumn("max_speed_kmh",           F.max("speed_from_prev_kmh").over(w_full)) \
    .withColumn("avg_speed_kmh",
        F.when(F.col("segment_count") == 0, F.lit(0.0))
         .otherwise(F.sum("speed_from_prev_kmh").over(w_full) / F.col("segment_count"))) \
    .withColumn("start_at_kst",            F.min("observed_at_kst").over(w_full)) \
    .withColumn("end_at_kst",              F.max("observed_at_kst").over(w_full))

In [ ]:
# 셀 9 - 일별·월별
bird_period = bird_agg \
    .withColumn("daily_distance_km",     F.sum("distance_from_prev_km").over(w_day)) \
    .withColumn("daily_avg_speed_kmh",   F.avg("speed_from_prev_kmh").over(w_day)) \
    .withColumn("daily_max_speed_kmh",   F.max("speed_from_prev_kmh").over(w_day)) \
    .withColumn("monthly_distance_km",   F.sum("distance_from_prev_km").over(w_month)) \
    .withColumn("monthly_avg_speed_kmh", F.avg("speed_from_prev_kmh").over(w_month)) \
    .withColumn("monthly_max_speed_kmh", F.max("speed_from_prev_kmh").over(w_month))

In [ ]:
# 셀 10 - 플래그 + season
bird_final = bird_period \
    .withColumn("is_first_point",      F.col("prev_point_order") == 0) \
    .withColumn("has_segment",         F.col("point_count") > 1) \
    .withColumn("has_speed_from_prev", F.col("speed_from_prev_kmh") > 0) \
    .withColumn("has_daily_speed",     F.col("daily_avg_speed_kmh") > 0) \
    .withColumn("has_monthly_speed",   F.col("monthly_avg_speed_kmh") > 0) \
    .withColumn("season_kst",
        F.when(F.col("observed_month_kst").isin(3,4,5),   "spring")
         .when(F.col("observed_month_kst").isin(6,7,8),   "summer")
         .when(F.col("observed_month_kst").isin(9,10,11), "autumn")
         .otherwise("winter")) \
    .withColumn("INGESTED_AT", F.current_timestamp())

In [ ]:
# 셀 11 - 저장
COLS = [
    "recv_id","taxon_nm","visitant_sno",
    "point_order","observed_timestamp_ms",
    "observed_at_utc","observed_at_kst","observed_date_kst",
    "observed_year_kst","observed_month_kst","observed_year_month_kst",
    "longitude","latitude","feature_1","feature_2",
    "point_count","segment_count","start_at_kst","end_at_kst",
    "total_distance_km","avg_segment_distance_km","max_segment_distance_km",
    "avg_speed_kmh","max_speed_kmh",
    "prev_point_order","distance_from_prev_km",
    "time_diff_from_prev_hours","speed_from_prev_kmh",
    "from_longitude","from_latitude","to_longitude","to_latitude",
    "daily_distance_km","daily_avg_speed_kmh","daily_max_speed_kmh",
    "monthly_distance_km","monthly_avg_speed_kmh","monthly_max_speed_kmh",
    "is_first_point","has_segment","has_speed_from_prev",
    "has_daily_speed","has_monthly_speed",
    "season_kst","observed_weekday_kst","observed_hour_kst","INGESTED_AT",
]

bird_final.select(COLS).write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(SILVER_BIRD)